# HC18 nnU-Net v2 Test Set Inference (Fold 0)
- Predicts on official 335 HC18 test set images.
- Uses trained weights from `prantakumerpandit390/hc18-nnunet-f0-250ep`.
- Applies corrected ellipse extraction pipeline (cv2.fitEllipseAMS, angle convention [-pi/2, pi/2)).
- Backs up model weights (`checkpoint_final.pth` / `checkpoint_best.pth`, `plans.json`, `dataset.json`) to output.
- Generates `submission_nnunet_f0_raw.csv` and `submission_nnunet_f0_measfix.csv`.


In [ ]:
import os
from pathlib import Path

print("=" * 60)
print("KERNEL STARTUP: Inspecting /kaggle/input/* and verifying test set")
print("=" * 60)

entries = []
for p in Path("/kaggle/input").iterdir():
    if p.is_dir():
        for sub in list(p.iterdir())[:10]:
            entries.append(str(sub))
    else:
        entries.append(str(p))

print(f"Total top-level / second-level entries found: {len(entries)}")
print("First 20 entries under /kaggle/input/*:")
for e in entries[:20]:
    print(" ", e)

# Exact subpath from kaggle datasets files prantakumerpandit390/hc18-ultrasound
exact_subpath = Path("test_set/test_set")
test_src_dir = None
for base in Path("/kaggle/input").iterdir():
    candidate = base / exact_subpath
    if candidate.is_dir():
        test_src_dir = candidate
        break

assert test_src_dir is not None, f"Could not find exact subpath {exact_subpath} under any folder in /kaggle/input"
test_pngs = sorted([p for p in test_src_dir.iterdir() if p.name.endswith(".png") and not p.name.endswith("_Annotation.png")])
print(f"Discovered test directory: {test_src_dir}")
print(f"Test PNG count: {len(test_pngs)}")
assert len(test_pngs) == 335, f"Expected exactly 335 test PNGs, got {len(test_pngs)}"
print("FAIL-FAST CHECK PASSED: Exactly 335 test images verified before proceeding.")


In [ ]:
!nvidia-smi
!pip install --upgrade pip
!pip install --quiet nnunetv2


In [ ]:
import os
import shutil
from pathlib import Path
import json

os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"

raw_dir = Path("/kaggle/working/nnUNet_raw")
prep_dir = Path("/kaggle/working/nnUNet_preprocessed")
res_dir = Path("/kaggle/working/nnUNet_results")
out_dir = Path("/kaggle/working/out")
backup_dir = Path("/kaggle/working/backup_weights")

for d in [raw_dir, prep_dir, res_dir, out_dir, backup_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Locating trained nnU-Net model weights in /kaggle/input...")
model_src_fold0 = None
for p in Path("/kaggle/input").rglob("fold_0"):
    if (p / "checkpoint_final.pth").exists() or (p / "checkpoint_best.pth").exists():
        model_src_fold0 = p
        break

if model_src_fold0 is None:
    for p in Path("/kaggle/input").rglob("checkpoint_best.pth"):
        model_src_fold0 = p.parent
        break

assert model_src_fold0 is not None, "Could not find fold_0 directory in /kaggle/input"
print(f"Found trained fold_0 at: {model_src_fold0}")

trainer_src = model_src_fold0.parent
plans_src = trainer_src / "plans.json"
dataset_src = trainer_src / "dataset.json"

target_trainer_dir = res_dir / "Dataset501_HC18" / trainer_src.name
target_fold0_dir = target_trainer_dir / "fold_0"
target_fold0_dir.mkdir(parents=True, exist_ok=True)

for f in model_src_fold0.glob("*"):
    if f.is_file():
        shutil.copy2(f, target_fold0_dir / f.name)
        print(f"  Staged {f.name} ({f.stat().st_size} bytes)")

if plans_src.exists():
    shutil.copy2(plans_src, target_trainer_dir / "plans.json")
    print(f"  Staged plans.json ({plans_src.stat().st_size} bytes)")
if dataset_src.exists():
    shutil.copy2(dataset_src, target_trainer_dir / "dataset.json")
    print(f"  Staged dataset.json ({dataset_src.stat().st_size} bytes)")

# Backup essential weights to backup_dir
for ckpt_name in ["checkpoint_final.pth", "checkpoint_best.pth"]:
    src_ckpt = target_fold0_dir / ckpt_name
    if src_ckpt.exists():
        shutil.copy2(src_ckpt, backup_dir / ckpt_name)
        print(f"Backed up {ckpt_name} to {backup_dir}")

if (target_trainer_dir / "plans.json").exists():
    shutil.copy2(target_trainer_dir / "plans.json", backup_dir / "plans.json")
if (target_trainer_dir / "dataset.json").exists():
    shutil.copy2(target_trainer_dir / "dataset.json", backup_dir / "dataset.json")

print("Backup contents:", list(backup_dir.glob("*")))


In [ ]:
import shutil
from pathlib import Path

test_images = sorted([p for p in test_src_dir.iterdir() if p.name.endswith(".png") and not p.name.endswith("_Annotation.png")])
print(f"Staging {len(test_images)} test images from {test_src_dir}...")
assert len(test_images) == 335, f"Expected 335 test images, got {len(test_images)}"

nnunet_test_in = Path("/kaggle/working/nnunet_test_in")
nnunet_test_in.mkdir(parents=True, exist_ok=True)

for img_path in test_images:
    case_id = img_path.stem
    dest_name = f"{case_id}_0000.png"
    shutil.copy2(img_path, nnunet_test_in / dest_name)

n_staged = len(list(nnunet_test_in.glob("*.png")))
print(f"Staged {n_staged} formatted test images in {nnunet_test_in}")
assert n_staged == 335


In [ ]:
import subprocess
from pathlib import Path

nnunet_test_out = Path("/kaggle/working/test_predicted_masks")
nnunet_test_out.mkdir(parents=True, exist_ok=True)

fold0_dir = res_dir / "Dataset501_HC18" / trainer_src.name / "fold_0"
if (fold0_dir / "checkpoint_final.pth").exists():
    chk_flag = "-chk checkpoint_final.pth"
elif (fold0_dir / "checkpoint_best.pth").exists():
    chk_flag = "-chk checkpoint_best.pth"
else:
    raise FileNotFoundError(f"No checkpoint found in {fold0_dir}")

cmd = f"nnUNetv2_predict -i {nnunet_test_in} -o {nnunet_test_out} -d 501 -c 2d -f 0 -tr {trainer_src.name} {chk_flag}"
print("Executing nnUNetv2_predict command:\n", cmd)
ret = subprocess.run(cmd, shell=True)
if ret.returncode != 0:
    raise RuntimeError(f"nnUNetv2_predict failed with return code {ret.returncode}")

predicted_masks = list(nnunet_test_out.glob("*.png"))
print(f"Prediction finished successfully: {len(predicted_masks)} masks generated.")
assert len(predicted_masks) == 335, f"Expected 335 masks, got {len(predicted_masks)}"


In [ ]:
import io
import cv2
import numpy as np
import pandas as pd
import hashlib
from pathlib import Path

# Robust ellipse fit
def fit_ellipse_robust(contour):
    try:
        return cv2.fitEllipseAMS(contour)
    except cv2.error:
        return cv2.fitEllipse(contour)

def hc_bpd_from_ellipse_params(semi_major_mm, semi_minor_mm):
    a, b = semi_major_mm, semi_minor_mm
    hc = np.pi * (3 * (a + b) - np.sqrt((3 * a + b) * (a + 3 * b)))
    bpd = 2 * min(a, b)
    return hc, bpd

def fit_submission_ellipse(prob_map_native, pixel_size, filename):
    binary = (prob_map_native > 0.5).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return {
            "filename": filename,
            "center_x_mm": 0.0,
            "center_y_mm": 0.0,
            "semi_axes_a_mm": 0.0,
            "semi_axes_b_mm": 0.0,
            "angle_rad": 0.0,
        }
    largest = max(contours, key=cv2.contourArea)
    if len(largest) < 5:
        return {
            "filename": filename,
            "center_x_mm": 0.0,
            "center_y_mm": 0.0,
            "semi_axes_a_mm": 0.0,
            "semi_axes_b_mm": 0.0,
            "angle_rad": 0.0,
        }
    (cx_px, cy_px), (axis1_px, axis2_px), angle_deg = fit_ellipse_robust(largest)
    semi_major_mm = (max(axis1_px, axis2_px) / 2.0) * pixel_size
    semi_minor_mm = (min(axis1_px, axis2_px) / 2.0) * pixel_size
    center_x_mm = cx_px * pixel_size
    center_y_mm = cy_px * pixel_size
    raw_angle_rad = np.deg2rad(angle_deg) - np.pi / 2.0
    angle_rad = float((raw_angle_rad + np.pi / 2.0) % np.pi - np.pi / 2.0)

    return {
        "filename": filename,
        "center_x_mm": float(center_x_mm),
        "center_y_mm": float(center_y_mm),
        "semi_axes_a_mm": float(semi_major_mm),
        "semi_axes_b_mm": float(semi_minor_mm),
        "angle_rad": float(angle_rad),
    }

# Load test pixel size data
csv_embedded = """filename,pixel size(mm)
000_HC.png,0.23585714045
001_HC.png,0.0679729758082
002_HC.png,0.165202488502
003_HC.png,0.0966742061652
004_HC.png,0.202202796936
005_HC.png,0.116687430892
006_HC.png,0.139810161458
007_HC.png,0.141644256456
008_HC.png,0.153437599991
009_HC.png,0.348563194275
010_HC.png,0.117421344269
011_HC.png,0.142794345106
012_HC.png,0.325923735096
013_HC.png,0.126876136925
014_HC.png,0.13266252844
015_HC.png,0.235724051793
016_HC.png,0.0914196751334
017_HC.png,0.128553937643
018_HC.png,0.147366365966
019_HC.png,0.100229513645
020_HC.png,0.0597069589865
021_HC.png,0.0700399951196
022_HC.png,0.134217634394
023_HC.png,0.113278505477
024_HC.png,0.128474204968
025_HC.png,0.119318962097
026_HC.png,0.106395204017
027_HC.png,0.182424241846
028_HC.png,0.121450729486
029_HC.png,0.118127706505
030_HC.png,0.104226768017
031_HC.png,0.113676948981
032_HC.png,0.109812967155
033_HC.png,0.140191747083
034_HC.png,0.127838827096
035_HC.png,0.119606271386
036_HC.png,0.260283110978
037_HC.png,0.0723095518955
038_HC.png,0.110730674532
039_HC.png,0.196812059365
040_HC.png,0.126094713807
041_HC.png,0.116940542709
042_HC.png,0.174222033599
043_HC.png,0.167659819126
044_HC.png,0.0950029436147
045_HC.png,0.0657061957999
046_HC.png,0.118081633534
047_HC.png,0.146053915515
048_HC.png,0.189785991724
049_HC.png,0.118193108411
050_HC.png,0.219291033952
051_HC.png,0.126872213581
052_HC.png,0.150450043606
053_HC.png,0.108915800634
054_HC.png,0.0973103969705
055_HC.png,0.112665422847
056_HC.png,0.112317679973
057_HC.png,0.125135138631
058_HC.png,0.166899820169
059_HC.png,0.148946327322
060_HC.png,0.146625296158
061_HC.png,0.209502403935
062_HC.png,0.113731595603
063_HC.png,0.112942684902
064_HC.png,0.183807037495
065_HC.png,0.202188564069
066_HC.png,0.113579861142
067_HC.png,0.134791699616
068_HC.png,0.235921258018
069_HC.png,0.119075633231
070_HC.png,0.124270230532
071_HC.png,0.0834785898527
072_HC.png,0.12214957214
073_HC.png,0.107152994643
074_HC.png,0.127284114177
075_HC.png,0.137042048815
076_HC.png,0.142642089299
077_HC.png,0.110157410304
078_HC.png,0.0629605278373
079_HC.png,0.217357342658
080_HC.png,0.121401243094
081_HC.png,0.160488188267
082_HC.png,0.107926079022
083_HC.png,0.112690751472
084_HC.png,0.111752642526
085_HC.png,0.14288854599
086_HC.png,0.129804371239
087_HC.png,0.185904657399
088_HC.png,0.322808219541
089_HC.png,0.131706501308
090_HC.png,0.104674311976
091_HC.png,0.104619053503
092_HC.png,0.281704983241
093_HC.png,0.106810952755
094_HC.png,0.122550275268
095_HC.png,0.12375841906
096_HC.png,0.161284256366
097_HC.png,0.124121285975
098_HC.png,0.140734457634
099_HC.png,0.250409305096
100_HC.png,0.106779552521
101_HC.png,0.134877361156
102_HC.png,0.123860173755
103_HC.png,0.104340277612
104_HC.png,0.126734172242
105_HC.png,0.281081795692
106_HC.png,0.166989525159
107_HC.png,0.109552119089
108_HC.png,0.107504826911
109_HC.png,0.119453862309
110_HC.png,0.219169725542
111_HC.png,0.0835672517618
112_HC.png,0.123499849947
113_HC.png,0.109908036564
114_HC.png,0.129494300255
115_HC.png,0.163182923871
116_HC.png,0.16132827728
117_HC.png,0.0629430413246
118_HC.png,0.0795051313582
119_HC.png,0.0714828201703
120_HC.png,0.0919303794702
121_HC.png,0.166956146558
122_HC.png,0.146154550945
123_HC.png,0.0569645315409
124_HC.png,0.122483881508
125_HC.png,0.100279283524
126_HC.png,0.0951051599575
127_HC.png,0.0736685184871
128_HC.png,0.123055475514
129_HC.png,0.123184352386
130_HC.png,0.112767768177
131_HC.png,0.1128863611
132_HC.png,0.0950197453769
133_HC.png,0.147917323253
134_HC.png,0.188716425086
135_HC.png,0.143880946296
136_HC.png,0.114491974766
137_HC.png,0.11803401368
138_HC.png,0.146099074798
139_HC.png,0.119778144927
140_HC.png,0.123034395823
141_HC.png,0.115126628767
142_HC.png,0.133844179076
143_HC.png,0.137925181124
144_HC.png,0.0659558412276
145_HC.png,0.104967442652
146_HC.png,0.124706700444
147_HC.png,0.0895000036274
148_HC.png,0.102403334209
149_HC.png,0.130508479319
150_HC.png,0.140730092223
151_HC.png,0.260221246001
152_HC.png,0.251992106438
153_HC.png,0.0608067541588
154_HC.png,0.132082575246
155_HC.png,0.119149869397
156_HC.png,0.248876139522
157_HC.png,0.0978129751542
158_HC.png,0.11566634095
159_HC.png,0.122087525158
160_HC.png,0.115348835324
161_HC.png,0.134699956791
162_HC.png,0.247320353985
163_HC.png,0.0717299069677
164_HC.png,0.11322368275
165_HC.png,0.115773255049
166_HC.png,0.112081739638
167_HC.png,0.104094830652
168_HC.png,0.109789190085
169_HC.png,0.119639337063
170_HC.png,0.0836038986842
171_HC.png,0.144835795675
172_HC.png,0.081993654126
173_HC.png,0.221762153837
174_HC.png,0.21740861561
175_HC.png,0.0632030111325
176_HC.png,0.146362229305
177_HC.png,0.0627333298326
178_HC.png,0.113191977143
179_HC.png,0.0744092019636
180_HC.png,0.133961232503
181_HC.png,0.128517808058
182_HC.png,0.100893330574
183_HC.png,0.184895065096
184_HC.png,0.10181704346
185_HC.png,0.1373954614
186_HC.png,0.191873449546
187_HC.png,0.0649442068942
188_HC.png,0.104747600853
189_HC.png,0.112921347779
190_HC.png,0.114284035834
191_HC.png,0.157119035721
192_HC.png,0.253724267211
193_HC.png,0.260235953641
194_HC.png,0.116326393083
195_HC.png,0.106863227296
196_HC.png,0.240968882319
197_HC.png,0.12634447217
198_HC.png,0.146495095071
199_HC.png,0.120391414826
200_HC.png,0.187674893273
201_HC.png,0.134385300649
202_HC.png,0.147598333219
203_HC.png,0.235888149057
204_HC.png,0.118606565964
205_HC.png,0.178828345878
206_HC.png,0.166479885578
207_HC.png,0.114593085918
208_HC.png,0.263994524353
209_HC.png,0.0974172470616
210_HC.png,0.170049970433
211_HC.png,0.260389625252
212_HC.png,0.0816284144511
213_HC.png,0.0790616254958
214_HC.png,0.133007356995
215_HC.png,0.13176147875
216_HC.png,0.102925166792
217_HC.png,0.0853389804646
218_HC.png,0.13478451484
219_HC.png,0.0846725247674
220_HC.png,0.120054951736
221_HC.png,0.111366669337
222_HC.png,0.136074874852
223_HC.png,0.122345555119
224_HC.png,0.0877106973999
225_HC.png,0.103067485328
226_HC.png,0.108590884053
227_HC.png,0.13682605447
228_HC.png,0.136454685314
229_HC.png,0.10873015808
230_HC.png,0.108923458535
231_HC.png,0.250230014324
232_HC.png,0.136471036318
233_HC.png,0.120521554028
234_HC.png,0.0701936617704
235_HC.png,0.0924945539898
236_HC.png,0.125404417515
237_HC.png,0.134797297619
238_HC.png,0.132069697506
239_HC.png,0.176806726626
240_HC.png,0.102872264629
241_HC.png,0.258437853593
242_HC.png,0.121416507698
243_HC.png,0.156431589276
244_HC.png,0.279805925157
245_HC.png,0.170023643364
246_HC.png,0.138377894958
247_HC.png,0.326014834066
248_HC.png,0.127538457895
249_HC.png,0.235924848488
250_HC.png,0.127579080753
251_HC.png,0.144020523344
252_HC.png,0.115545288075
253_HC.png,0.116019935109
254_HC.png,0.253417733349
255_HC.png,0.143333332879
256_HC.png,0.0752148343556
257_HC.png,0.0716604845864
258_HC.png,0.128584733376
259_HC.png,0.0655996501446
260_HC.png,0.120866364095
261_HC.png,0.136534458882
262_HC.png,0.0691229436133
263_HC.png,0.258141603225
264_HC.png,0.326155078027
265_HC.png,0.348773701438
266_HC.png,0.114554193887
267_HC.png,0.150791623376
268_HC.png,0.194670901849
269_HC.png,0.117122087368
270_HC.png,0.139655884769
271_HC.png,0.134715220413
272_HC.png,0.119597074531
273_HC.png,0.111617008845
274_HC.png,0.157924108207
275_HC.png,0.116871040921
276_HC.png,0.133633346934
277_HC.png,0.112222221163
278_HC.png,0.0839414397875
279_HC.png,0.101253809978
280_HC.png,0.258454390061
281_HC.png,0.166432291269
282_HC.png,0.141810868468
283_HC.png,0.240856983575
284_HC.png,0.12864721127
285_HC.png,0.118230489038
286_HC.png,0.178931334189
287_HC.png,0.0659489631653
288_HC.png,0.0851170289314
289_HC.png,0.202066729767
290_HC.png,0.235798713707
291_HC.png,0.371304794594
292_HC.png,0.260168577169
293_HC.png,0.240741422377
294_HC.png,0.114064094695
295_HC.png,0.125862568617
296_HC.png,0.107785787271
297_HC.png,0.163124030636
298_HC.png,0.102101199481
299_HC.png,0.222413380941
300_HC.png,0.131601001087
301_HC.png,0.153206099163
302_HC.png,0.191834316804
303_HC.png,0.121687912359
304_HC.png,0.221209857199
305_HC.png,0.0525587044264
306_HC.png,0.125102937222
307_HC.png,0.137146066975
308_HC.png,0.240979280817
309_HC.png,0.108860342399
310_HC.png,0.0602671539927
311_HC.png,0.110341873798
312_HC.png,0.2953860339
313_HC.png,0.0831504066785
314_HC.png,0.140055881606
315_HC.png,0.190566801108
316_HC.png,0.186685941837
317_HC.png,0.0639305817775
318_HC.png,0.220662593842
319_HC.png,0.15726249665
320_HC.png,0.0601536574134
321_HC.png,0.121628409479
322_HC.png,0.109896463352
323_HC.png,0.147385439452
324_HC.png,0.0856688258977
325_HC.png,0.130876192921
326_HC.png,0.260200190854
327_HC.png,0.152073358044
328_HC.png,0.124448135495
329_HC.png,0.110538191266
330_HC.png,0.138479305638
331_HC.png,0.0757386377363
332_HC.png,0.113106918606
333_HC.png,0.219140493351
334_HC.png,0.112420544028"""
df_ps = pd.read_csv(io.StringIO(csv_embedded))
pixel_size_map = dict(zip(df_ps["filename"], df_ps["pixel size(mm)"]))
print(f"Loaded {len(pixel_size_map)} test pixel sizes.")
assert len(pixel_size_map) == 335

rows = []
hc_values = []
mask_dir = Path("/kaggle/working/test_predicted_masks")
test_fns = sorted(list(pixel_size_map.keys()))

for fn in test_fns:
    mask_file = mask_dir / fn
    assert mask_file.exists(), f"Missing predicted mask: {mask_file}"
    mask_img = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)
    prob_map = (mask_img > 0).astype(np.float32)
    ps = pixel_size_map[fn]
    rec = fit_submission_ellipse(prob_map, ps, fn)
    rows.append(rec)
    hc, _ = hc_bpd_from_ellipse_params(rec["semi_axes_a_mm"], rec["semi_axes_b_mm"])
    hc_values.append(hc)

df_sub = pd.DataFrame(rows)

# Strict column order
columns = ["filename", "center_x_mm", "center_y_mm", "semi_axes_a_mm", "semi_axes_b_mm", "angle_rad"]
df_sub = df_sub[columns]

# Rounding
for col in ["center_x_mm", "center_y_mm", "semi_axes_a_mm", "semi_axes_b_mm", "angle_rad"]:
    df_sub[col] = df_sub[col].round(6)

out_csv = Path("/kaggle/working/submission_nnunet_f0_raw.csv")
csv_bytes = df_sub.to_csv(index=False, lineterminator="\n").encode("utf-8")
out_csv.write_bytes(csv_bytes)

# Verify assertions
assert len(df_sub) == 335, f"Expected 335 rows, got {len(df_sub)}"
assert df_sub["filename"].nunique() == 335, "Duplicate filenames found"
assert df_sub["angle_rad"].min() >= -np.pi / 2.0 - 1e-5, "Angle below -pi/2"
assert df_sub["angle_rad"].max() <= np.pi / 2.0 + 1e-5, "Angle above pi/2"
assert b"\r" not in csv_bytes, "CRLF detected"

sha256 = hashlib.sha256(csv_bytes).hexdigest()
print("=" * 60)
print("SUCCESSFULLY GENERATED RAW SUBMISSION CSV:")
print(f"  File  : {out_csv}")
print(f"  Rows  : {len(df_sub)}")
print(f"  SHA256: {sha256}")
print(f"  Mean HC on test set: {np.mean(hc_values):.4f} mm (std: {np.std(hc_values):.4f} mm)")
print(f"  Mean semi_axes_a: {df_sub['semi_axes_a_mm'].mean():.4f} mm")
print(f"  Mean semi_axes_b: {df_sub['semi_axes_b_mm'].mean():.4f} mm")
print("First 3 rows:")
print(df_sub.head(3))
print("Last 3 rows:")
print(df_sub.tail(3))
print("=" * 60)

# Build measfix candidate (semi-axes shrunk by delta_r = 0.113422 mm)
delta_r = 0.11342152722878392
df_meas = df_sub.copy()
df_meas["semi_axes_a_mm"] = (df_sub["semi_axes_a_mm"] - delta_r).clip(lower=0.1).round(6)
df_meas["semi_axes_b_mm"] = (df_sub["semi_axes_b_mm"] - delta_r).clip(lower=0.1).round(6)

out_meas_csv = Path("/kaggle/working/submission_nnunet_f0_measfix.csv")
meas_bytes = df_meas.to_csv(index=False, lineterminator="\n").encode("utf-8")
out_meas_csv.write_bytes(meas_bytes)

sha256_meas = hashlib.sha256(meas_bytes).hexdigest()
print("SUCCESSFULLY GENERATED MEASFIX SUBMISSION CSV:")
print(f"  File  : {out_meas_csv}")
print(f"  Rows  : {len(df_meas)}")
print(f"  SHA256: {sha256_meas}")
print("=" * 60)


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/kaggle/working/test_predicted_masks.zip")
print(f"Creating zip archive {zip_path}...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for m in Path("/kaggle/working/test_predicted_masks").glob("*.png"):
        zf.write(m, arcname=m.name)
print(f"Archive created successfully: {zip_path.stat().st_size / 1024 / 1024:.2f} MB")
print("Task B nnU-Net test inference completed successfully.")
